# Network Expansion
## Model 3 - Uncertainty-Aware / Robust Extension

This notebook outlines the Model 3 formulation (scenario-based two-stage) building on the multi-year Model 2. First-stage investment decisions (substation activation, feeder lines, capacity reinforcements) are shared across scenarios; second-stage operational flows/assignments are scenario-specific. Objective options: expected discounted cost or worst-case (robust/CVaR) with reliability enforced per scenario.

### Roadmap
1. Set up the base distribution network (same topology as Model 1/2).
2. Define multi-year demand scenarios with probabilities.
3. Package data for a stochastic/robust solver (`solve_network_stochastic`, to be added in `src/solver.py`).
4. Inspect first-stage decisions vs. scenario-dependent operations once the solver is implemented.

### 1 - Imports

In [ ]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
# from src.solver import solve_network_stochastic  # TODO: implement in src/solver.py

### 2 - Define the Distribution Network (shared across scenarios)

In [ ]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial (online) substation
S1 = Substation("S1", "N4", 40, ["N3", "N5", "N9"], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50  # Default cost of building network lines (non-substation feeders)

# Base load (peak) demand
base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

# Load locations
loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

# Undirected connectivity (will be doubled to directed arcs)
nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5', 'N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

# Instantiate network
DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations (same as Model 1/2)
capacity = 15
s_cost = 100       # Cost of substation activation
l_cost = line_cost # Cost of connecting a substation feeder line
r_cost = 200       # Cost of capacity reinforcement

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])


### 3 - Multi-year demand scenarios

In [ ]:
# Horizon and parameters
years = list(range(1, 11))
R = 10        # Size of a single capacity reinforcement
B = 550       # Annual budget (nominal)
dr = 0.05     # Discount rate

# Scenario definitions: probability and annual growth multipliers
# Example: conservative keeps demand flat, base grows moderately, high grows faster
scenarios = {
    'conservative': {
        'prob': 0.25,
        'growth': [1.00] + [1.01] * 9
    },
    'base': {
        'prob': 0.5,
        'growth': [1.00] + [1.04] * 9
    },
    'high': {
        'prob': 0.25,
        'growth': [1.00] + [1.07] * 9
    }
}

# Build scenario-specific demand tensors per year and load
scenario_demands = {}
for omega, data in scenarios.items():
    growth = data['growth']
    scenario_demands[omega] = {}
    for y in years:
        factor = growth[y-1]
        scenario_demands[omega][y] = {ld: cap * factor for ld, cap in base_load_capacity.items()}

# Assemble data package for the stochastic/robust solver (to be implemented)
model_3_data = {
    'network': DistributionNetwork,
    'years': years,
    'scenarios': scenarios,
    'scenario_demands': scenario_demands,
    'budget': B,
    'reinforcement_size': R,
    'discount_rate': dr
}

model_3_data


### 4 - Solver call (placeholder)
Once `solve_network_stochastic` is implemented in `src/solver.py`, call it with the packaged data to optimize investments (first stage) and operations (second stage) under uncertainty.

```python
# TODO: implement solve_network_stochastic(...) in src/solver.py
# solution = solve_network_stochastic(model_3_data)
# print(solution['first_stage'])
# print(solution['scenario_ops']['high'])
```